# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
aviation_df = pd.read_csv(
    'data/AviationData.csv',
    encoding='cp1252',
    low_memory=False
)

# view the df at a glance
pd.set_option('display.max_columns', None)
aviation_df.head()

# check the available keys to see what's available.
aviation_df.keys()

# Intial thoughts based on looking at data:
# - Amatuer.Built may help determine if build is professional
# - Model may help determine if build is active from 1983 and onwards

# Investigate missing values
# aviation_df.isna().sum()

aviation_df.head()



/var/folders/42/wx6_spn57w708fqcmzgcrnd80000gn/T/ipykernel_23559/806691131.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv('data/AviationData.csv', encoding='cp1252')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# inital row cleaning to ensure rows are consistent capitalized and there is no whitespace
aviation_df['Air.carrier'] = aviation_df['Air.carrier'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.upper()
aviation_df['Airport.Name'] = aviation_df['Airport.Name'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.upper()
aviation_df['Location'] = aviation_df['Location'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.upper()
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.lower()
aviation_df['Weather.Condition'] = aviation_df['Weather.Condition'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.lower()
aviation_df['Purpose.of.flight'] = aviation_df['Purpose.of.flight'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.lower()
aviation_df['Broad.phase.of.flight'] = aviation_df['Broad.phase.of.flight'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.lower()
aviation_df['Number.of.Engines'] = aviation_df['Number.of.Engines'].replace(r'[^A-Za-z0-9\s]', '', regex=True).round().astype('Int64')

# filter out any amature built aircrafts
aviation_df_filtered = aviation_df[aviation_df['Amateur.Built'].str.strip().str.lower() == 'no']

# filter out planes before 1983
aviation_df_filtered = aviation_df_filtered[pd.to_datetime(aviation_df_filtered['Event.Date']) >= '1983-01-01']

# ensure it is an airplane
aviation_df_filtered = aviation_df_filtered[aviation_df_filtered['Aircraft.Category'].str.strip().str.lower() == 'airplane']



### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# create an estimated total of all individuals based on fatal + serious + minor + uninjured columns
aviation_df_filtered['Total.Estimated.People'] = aviation_df_filtered[
    ['Total.Fatal.Injuries',
     'Total.Serious.Injuries',
     'Total.Minor.Injuries','Total.Uninjured']
].sum(axis=1)


aviation_df_filtered['Injury.Percentage'] = (
    (aviation_df_filtered['Total.Serious.Injuries'] +
     aviation_df_filtered['Total.Fatal.Injuries'])
    / aviation_df_filtered['Total.Estimated.People']
)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
# ensure damage columns are consistent and create a new column to determine if an airfcraft was destroyed
aviation_df_filtered['Aircraft.damage'] = aviation_df_filtered['Aircraft.damage'].str.strip().str.lower()
aviation_df_filtered['Aircraft.destroyed'] = aviation_df_filtered['Aircraft.damage'] == 'destroyed'

# filter destroyed aircrafts 
destroyed_aircrafts =  aviation_df_filtered[aviation_df_filtered['Aircraft.destroyed'] == True]

destroyed_aircrafts.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Estimated.People,Injury.Percentage,Aircraft.destroyed
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,CROSSVILLE TN,United States,NaN,NaN,NaN,NaN,Fatal(1),destroyed,Airplane,N9600W,Piper,PA-28-140,No,1,reciprocating,Part 91: General Aviation,NaN,personal,NaN,1.0,1.0,NaN,NaN,imc,cruise,Probable Cause,02-05-2011,2.0,1.0,True
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,SILVANA WA,United States,NaN,NaN,S88,NaN,Fatal(11),destroyed,Airplane,N116CA,Lockheed,"LEARSTAR, L-18-56",No,2,reciprocating,Part 91: General Aviation,NaN,skydiving,NaN,11.0,2.0,NaN,13.0,vmc,other,Probable Cause,17-10-2016,26.0,0.5,True
8865,20001214X40407,Accident,MKC84FA197,1984-07-03,WRIGHT AR,United States,NaN,NaN,NaN,NaN,Fatal(1),destroyed,Airplane,N4025,Piper,PA-18-150,No,1,reciprocating,Part 137: Agricultural,UNK,aerial application,NaN,1.0,NaN,NaN,NaN,vmc,maneuvering,Probable Cause,15-12-2009,1.0,NaN,True
10605,20001214X41706,Accident,ATL85FA072,1984-12-30,DUBLIN VA,United States,NaN,NaN,PSK,NEW RIVER VALLEY,Fatal(1),destroyed,Airplane,N4963D,Cessna,182A,No,1,reciprocating,Part 91: General Aviation,NaN,skydiving,NaN,1.0,NaN,NaN,NaN,vmc,maneuvering,Probable Cause,17-10-2016,1.0,NaN,True
10688,20001214X35509,Accident,DEN85LA064,1985-01-14,WAPITI WY,United States,NaN,NaN,NaN,NaN,Non-Fatal,destroyed,Airplane,N759WB,Cessna,182Q,No,1,reciprocating,Part 91: General Aviation,NaN,personal,NaN,NaN,1.0,1.0,NaN,vmc,maneuvering,Probable Cause,12-01-2016,2.0,NaN,True


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
# update Make names for consistency. Remove spaces and uppercase 
aviation_df_filtered['Make'] = aviation_df_filtered['Make'].replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip().str.upper()

# stadardize make names
make_mapping = {

    'ABC CORP': 'ABC',
    'ABC CORPORATION': 'ABC',
    'ABC INC': 'ABC',
    'AERONCA': 'AERONCA',
    '737':'737',
    'AERO': 'AERO',
    'AYRES':'AYRES',
    'ANTONOV':'ANTONOV',
    'AMERICAN LEGAND':'AMERICAN LEGAND',
    'APOLLO':'APOLLO',
    'AIR TRACTOR': 'AIR TRACTOR',
    'AIRCRAFT MFG': 'AIRCRAFT MFG',
    'AIRPLANE FACTORY': 'AIRPLANE FACTORY',
    'QUICKSILVER AIRCRAFT': 'QUICKSILVER AIRCRAFT',
    'AMERICAN CHAMPION': 'AMERICAN CHAMPION',
    'WSK': 'WSK',
    'ZENAIR': 'ZENAIR',
    'ZENITH': 'ZENITH',
    'ZLIN': 'ZLIN',
    'WINGTIP': 'WINGTIP',
    'YAKOVLEV': 'YAKOVLEV',
    'WEATHERLY': 'WEATHERLY',
    'VANS': 'VANS',
    'WACO': 'WACO',
    'UNIVAIR': 'UNIVAIR',
    'THRUSH': 'THRUSH',
    'SKYBOLT': 'SKYKITS',
    'SKYKITS CORP': 'SKYKITS',
    'SKYKITS CORPORATION': 'SKYKITS',
    'SKYKITS USA CORP': 'SKYKITS',
    'AMERICAN LEGEND': 'AMERICAN LEGEND',
    'EMBRAER': 'EMBRAER',
    'TAYLOR': 'TAYLOR',
    'TEXTRON': 'TEXTRON',
    'AIRBUS': 'AIRBUS',
    'SHORT BROS': 'SHORT BROS'
    'VARGA, VARGA',
    'STORCH':'STORCH',
    'STOL':'STOL',
    'STEARMAN':'STEARMAN',
    'SCOTTISH':'SCOTTISH',
    'SCHWEIZER':'SCHWEIZER',
    'SAAB':'SAAB',
    'ROCKWELL':'ROCKWELL',
    'REMOS':'REMOS',
    'REIMS':'REIMS',
    'RAYTHEON':'RAYTHEON',
    'RANS':'RANS',
    'RAINBOW' :'RAINBOW' ,
    'QUICKSILVER':'QUICKSILVER',
    'QUEST':'QUEST',
    'QUAD':'QUAD',
    'PZL':'PZL',
    'PIPISTREL':'PIPISTREL',
    'PIPER':'PIPER',
    'PILATUS':'PILATUS',
    'PIAGGIO':'PIAGGIO',
    'PHANTOM':'PHANTOM',
    'PARTENAVIA':'PARTENAVIA',
    'PARADISE':'PARADISE',
    'NORTH WING':'NORTH WING',
    'NEW PIPER':'NEW PIPER',
    'MOYES':'MOYES',
    'MSQUARED':'MSQUARED',
    'MOONEY':'MOONEY',
    'MONOCOUPE':'MONOCOUPE',
    'MAXAIR':'MAXAIR',
    'CIRRUS':'CIRRUS',
    'CESSNA':'CESSNA',
    'CENTRAL OHIO DRAGONFLY':'CENTRAL OHIO DRAGONFLY',
    'CANADAIR':'CANADAIR',
    'BRITISH AIRCRAFT CORP':'BRITISH AIRCRAFT CORP',
    'JABIRU':'JABIRU',
    'HOWARD':'HOWARD',
    'HONDA':'HONDA',
    'DIAMOND':'DIAMOND',
    'MCDONNELL DOUGLAS':'MCDONNELL DOUGLAS',
    'DOUGLAS':'MCDONNELL DOUGLAS',
    'BOEING':'BOEING',
    'SHORT BROTHERS':'SHORT BROS',
    'INDUS AVIATION':'INDUS AVIATION',
    'GRUMMAN':'GRUMMAN',
    'BOMBARDIER':'BOMBARDIER',
    'ALASKAN':'ALASKAN',
    'AIRBORNE':'AIRBORNE',
    'HAWKER':'HAWKER',
    'BEECH':'BEECH',
    'TL ULTRALIGHT':'TL ULTRALIGHT',
    'T BIRD':'T BIRD',
    'SAABSCANIA':'SAABSCANIA',
    'UNKNOWN':'UNREGISTERED',
    
}

for pattern, replacement in make_mapping.items():
    mask = aviation_df_filtered['Make'].str.contains(
        pattern, case=False, na=False
    )
    aviation_df_filtered.loc[mask, 'Make'] = replacement


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# remove NaNs from model
aviation_df_filtered = aviation_df_filtered[
    aviation_df_filtered['Model'].notna()
]

# standerdize capitalization and spaces
aviation_df_filtered['Model'] = aviation_df_filtered['Model'].str.strip().replace(r'[^A-Za-z0-9]', '', regex=True).str.upper()

# see how often the make model combo surface
aviation_df_filtered[['Make', 'Model']].value_counts()

# see which vales appear together
model_make_counts = aviation_df_filtered.groupby('Model')['Make'].nunique()

model_make_counts[model_make_counts > 1]



Model
100            5
108            2
1081           3
1082           2
10A            2
              ..
V35B           2
XAIR           2
YAK52          2
ZENITH701      2
ZODIAC601XL    3
Name: Make, Length: 364, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# Already completed in earlier step (Data Cleaning)


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
aviation_df.isna().sum().sort_values(ascending=False)

aviation_df_filtered = aviation_df_filtered.drop(
    columns=[
        'Latitude',
        'Longitude',
        'Airport.Code',
        'Airport.Name',
        'Schedule',
        'Air.carrier',
    ]
)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
aviation_df_filtered.to_csv('cleaned_aviation_data', index=False) 